In [3]:
import json
from tqdm import tqdm

def read_game(filename):
    with open(filename, 'r') as f:
        game_set_data = json.load(f)

    filtered_games = {}
    unfinished = 0
    aborted = 0
    lobby = 0
    other = 0
    too_few_players = 0
    too_few_rounds = 0
    bots = 0
    no_chat = 0

    for game_id in tqdm(game_set_data):
        game_data = game_set_data[game_id]

        public_info = game_data['publicInfo']
        private_info = game_data['privateInfo']
        status = public_info['gameStatus'] 

        if status != 'finished':
            unfinished += 1
            if status == 'lobby':
                lobby +=1
            elif status == 'aborted':
                aborted += 1
            else:
                other += 1

            continue

        num_players = public_info['numPlayers']

        if num_players < 5:
            too_few_players += 1
            continue

        round_num = public_info['roundNum']

        if round_num < 5:
            too_few_rounds += 1
            continue

        bot_found = False

        players = private_info['players']
        for player in players:
            experience = players[player]['experience']
            if experience == "advanced":
                bot_found = True
                break

        if bot_found:
            bots += 1
            continue

        if public_info['gameParams']['chatType'] != 'direct':
            no_chat += 1
            continue

        filtered_games[game_id] = game_data

    print(f"Total filtered games: {len(filtered_games)}")
    print (f"Unfinished: {unfinished}  Aborted: {aborted}  Lobby:  {lobby}  Other: {other}")
    print(f"Too few players: {too_few_players}  Too few rounds:  {too_few_rounds}  Bots: {bots}  No chat: {no_chat}")

    save_file = filename.replace('.json', '_filtered.json')
    with open(save_file, 'w') as f:
        json.dump(filtered_games, f, indent=2)  

read_game('preprod.json')

100%|██████████| 5251/5251 [00:00<00:00, 768762.97it/s]


Total filtered games: 156
Unfinished: 1311  Aborted: 0  Lobby:  1243  Other: 68
Too few players: 2369  Too few rounds:  658  Bots: 644  No chat: 113


In [5]:
from pathlib import Path
import json
import tqdm

def extract_chat_info(
    input_file='preprod_filtered.json',
    output_folder='chat_info'
):
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    with open(input_file, 'r') as f:
        games = json.load(f)

    for game_id, game_data in tqdm.tqdm(games.items()):
        chat_info = game_data.get('chatInfo', {})

        output_file = output_folder / f'{game_id}.json'
        with open(output_file, 'w') as f:
            json.dump(chat_info, f, indent=2)

    print(f'Created {len(games)} chat files in "{output_folder}"')

extract_chat_info()

100%|██████████| 156/156 [00:00<00:00, 427.38it/s]

Created 156 chat files in "chat_info"


In [ ]:
# print all of the file names in chat_info folder
import os
import json

game_ids = []

chat_info_folder = 'chat_info'
for filename in os.listdir(chat_info_folder):
    if filename.endswith('.json'):
        if filename.endswith('.reasons.json') or filename.endswith('.stopped_reasons.json'):
            continue
        game_id = filename.replace('.json', '')
        game_ids.append(game_id)

print (f"Total chat files: {len(game_ids)}")
print (f"Game IDs: {game_ids}")

# filter preprod_filtered.json to only include the game_ids in chat_info folder
def filter_games_by_chat_info(input_file='preprod_filtered.json', output_file='preprod_filtered_chat_info.json', game_ids=None):
    if game_ids is None:
        game_ids = []

    with open(input_file, 'r') as f:
        games = json.load(f)

    filtered_games = {game_id: games[game_id] for game_id in game_ids if game_id in games}

    with open(output_file, 'w') as f:
        json.dump(filtered_games, f, indent=2)

    print(f"Filtered games saved to {output_file}. Total filtered games: {len(filtered_games)}")

filter_games_by_chat_info(game_ids=game_ids)

Total chat files: 41
Game IDs: ['5c1bded0-6697-1f77-afde-1199151f1ce5', '489a2d71-4c03-11ed-adcf-c96787d92db2', '84c50300-2f84-1257-8bc1-57f65e177c74', '6bf27f20-60d3-1442-ad65-5fb5f4d38319', '3f3bed11-6f9b-11ee-b368-6d3a9856e36a', '306c3230-f05c-145a-b467-cf1e073de8ae', 'c6875f11-4bf3-11ed-8571-c723ad250a98', 'bf8f1571-5bf3-11ee-af1a-4be82ae43316', 'e93abe51-b5e7-11f1-9217-114c99476a9f', '9522b6b1-79d1-11ee-b6fc-8ffc2153ff46', 'e93f4f61-6a7e-11f1-9949-716ca35cc062', 'c9905c61-049a-11ee-b7a4-0dea69b377fd', 'f7fa4401-4df7-11f0-b3ab-45e3dabcd1e4', '7c6801d1-4b19-11ed-9aa4-69346008512e', '679674d0-908a-1470-bfc7-6b2913e3fff4', '10a50171-f5ad-11ed-b447-611577d8ce81', 'e03a2cb1-9048-11f1-aa4f-e3292dced73f', '24c1f420-fb6a-1fb3-88b9-5fec06f6d1fa', '7e8f92c1-b5e6-11f1-9217-114c99476a9f', '9ffff481-9049-11f1-aa4f-e3292dced73f', 'c29e8351-ae23-11ed-90c3-47905af5efb2', '9ec38bc1-8c4a-11f1-992e-f12499d781ea', '67335941-b6f3-11ed-83d6-392ce5c31d49', '22e26001-496d-11f1-bc32-efaef376b110', '299dc86

In [3]:
import json

# filter preprod_filter.json to remove games with doctorGameNames, medSchoolGameNames, healPayments, or healRequests keys
def filter_games(input_file='preprod_filtered.json', output_file='preprod_filtered_no_medical.json'):
    with open(input_file, 'r') as f:
        games = json.load(f)

    filtered_games = {}
    removed_count = 0

    for game_id, game_data in games.items():
        if any(key in game_data for key in ['doctorGameNames', 'medSchoolGameNames', 'healPayments', 'healRequests']):
            removed_count += 1
            continue
        filtered_games[game_id] = game_data

    with open(output_file, 'w') as f:
        json.dump(filtered_games, f, indent=2)

    print(f"Filtered games saved to {output_file}. Total filtered games: {len(filtered_games)}. Removed games: {removed_count}")

filter_games()

Filtered games saved to preprod_filtered_no_medical.json. Total filtered games: 30. Removed games: 11
